# Демонстрация работы слоёв и контрактов Memory-Agents

Этот ноутбук показывает, как API-слой, бизнес-логика, доменные сервисы и слой хранения обмениваются контрактами на примере создания эпизода и записи знания.


## План

1. Определим упрощённые версии стабильных API-контрактов (`UniversalEpisodeWriteRequest`, `UniversalKnowledgeWriteRequest`).
2. Реализуем бизнес-слой с `RequestValidator` и `IdempotencyGuard`.
3. Опишем доменные сервисы и заглушки слоя хранения.
4. Соберём пайплайн и прогонем сценарии создания эпизода и знания, прослеживая данные на каждом шаге.
5. Сформируем единый журнал событий для аудита взаимодействий.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional
import json


@dataclass
class EpisodeContext:
    user_id: str
    agent_id: str
    session_id: Optional[str] = None
    team_id: Optional[str] = None


@dataclass
class EpisodeTrajectoryStep:
    step_id: str
    timestamp: datetime
    role: str
    action: str
    content: str
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class UniversalEpisodeWriteRequest:
    idempotency_key: str
    context: EpisodeContext
    scope: str
    episode_type: str
    title: str
    trajectory: List[EpisodeTrajectoryStep]
    outcome: Optional[str] = None
    importance: float = 0.5
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class UniversalKnowledgeWriteRequest:
    knowledge_id: str
    knowledge: str
    source: str
    confidence: float
    agent_id: str
    supporting_evidence: List[str] = field(default_factory=list)
    access_scope: str = "agent_private"


@dataclass
class UniversalWriteResponse:
    request_id: str
    status: str
    payload: Dict[str, Any]


def pretty(obj: Any) -> str:
    """Helper for JSON pretty-printing."""
    return json.dumps(obj, ensure_ascii=False, indent=2, default=str)


## Бизнес-слой: RequestValidator и IdempotencyGuard

Упростим гварды до минимальной логики:

- `RequestValidator` проверяет базовые поля и нормализует DTO.
- `IdempotencyGuard` хранит кеш ответов по ключу, имитируя Redis + PostgreSQL.

В реальной системе сюда добавляются правила доступа, rate limit и аудит.


In [ ]:
class RequestValidator:
    def validate_episode(self, request: UniversalEpisodeWriteRequest) -> UniversalEpisodeWriteRequest:
        assert request.context.user_id, "user_id обязателен"
        assert request.scope in {
            "user_private",
            "team_shared",
            "agent_private",
        }, "неизвестный scope"
        assert 0.0 <= request.importance <= 1.0, "важность должна быть 0..1"
        return request

    def validate_knowledge(self, request: UniversalKnowledgeWriteRequest) -> UniversalKnowledgeWriteRequest:
        assert len(request.knowledge) > 0, "knowledge не может быть пустым"
        assert 0.0 <= request.confidence <= 1.0, "confidence 0..1"
        return request


class IdempotencyGuard:
    def __init__(self):
        self._cache: Dict[str, UniversalWriteResponse] = {}

    def execute(self, key: str, handler) -> UniversalWriteResponse:
        if key in self._cache:
            cached = self._cache[key]
            print(f"[IdempotencyGuard] возврат из кеша для {key}")
            return cached
        response = handler()
        self._cache[key] = response
        print(f"[IdempotencyGuard] записали ответ для {key}")
        return response


guard = IdempotencyGuard()
validator = RequestValidator()


## Доменный слой и клиенты хранения

Ниже определим:

- `InMemoryMongo`, `InMemoryQdrant`, `InMemoryRedis` — роли хранилищ.
- Сервисы `EpisodicMemoryService` и `SemanticMemoryService`, работающие поверх клиентов.

Это позволит показать, какие структуры передаются на границе «домен ↔ storage».


In [ ]:
class InMemoryMongo:
    def __init__(self):
        self.documents: List[Dict[str, Any]] = []

    def insert(self, collection: str, document: Dict[str, Any]) -> str:
        document = {**document, "_collection": collection, "_id": len(self.documents) + 1}
        self.documents.append(document)
        print(f"[Mongo] insert into {collection}: {_safe(document)}")
        return document["_id"]

    def find(self, collection: str, query: Dict[str, Any]) -> List[Dict[str, Any]]:
        results = [doc for doc in self.documents if doc.get("_collection") == collection]
        print(f"[Mongo] find from {collection}: {query} -> {len(results)} docs")
        return results


class InMemoryQdrant:
    def __init__(self):
        self.vectors: List[Dict[str, Any]] = []

    def insert(self, collection: str, payload: Dict[str, Any]) -> None:
        payload = {**payload, "collection": collection}
        self.vectors.append(payload)
        print(f"[Qdrant] upsert into {collection}: {_safe(payload)}")


class InMemoryRedis:
    def __init__(self):
        self.cache: Dict[str, Any] = {}

    def set(self, key: str, value: Any) -> None:
        self.cache[key] = value
        print(f"[Redis] set {key}")


mongo = InMemoryMongo()
qdrant = InMemoryQdrant()
redis = InMemoryRedis()


def _safe(data: Dict[str, Any]) -> str:
    copy = {k: v for k, v in data.items() if not k.startswith("_")}
    return pretty(copy)


class EpisodicMemoryService:
    def __init__(self, mongo_client: InMemoryMongo, qdrant_client: InMemoryQdrant):
        self.mongo = mongo_client
        self.qdrant = qdrant_client

    def create_episode(self, dto: UniversalEpisodeWriteRequest) -> UniversalWriteResponse:
        document = {
            "episode_id": dto.idempotency_key,
            "context": dto.context.__dict__,
            "scope": dto.scope,
            "episode_type": dto.episode_type,
            "title": dto.title,
            "trajectory": [step.__dict__ for step in dto.trajectory],
            "importance": dto.importance,
            "metadata": dto.metadata,
        }
        doc_id = self.mongo.insert("episodes", document)
        self.qdrant.insert(
            "episodes",
            {
                "episode_id": dto.idempotency_key,
                "agent_id": dto.context.agent_id,
                "importance": dto.importance,
            },
        )
        return UniversalWriteResponse(
            request_id=str(doc_id),
            status="created",
            payload={"episode_id": dto.idempotency_key},
        )


class SemanticMemoryService:
    def __init__(self, mongo_client: InMemoryMongo):
        self.mongo = mongo_client

    def create_knowledge(self, dto: UniversalKnowledgeWriteRequest) -> UniversalWriteResponse:
        document = {
            "knowledge_id": dto.knowledge_id,
            "knowledge": dto.knowledge,
            "source": dto.source,
            "confidence": dto.confidence,
            "supporting_evidence": dto.supporting_evidence,
            "agent_id": dto.agent_id,
        }
        doc_id = self.mongo.insert("knowledge", document)
        return UniversalWriteResponse(
            request_id=str(doc_id),
            status="created",
            payload={"knowledge_id": dto.knowledge_id},
        )


episodic_service = EpisodicMemoryService(mongo, qdrant)
semantic_service = SemanticMemoryService(mongo)


## Сквозной пайплайн

Соберём координатор, который принимает универсальные запросы, прогоняет их через валидатор и идемпотентность, а затем вызывает соответствующий доменный сервис.


In [ ]:
class MemoryFacadeDemo:
    def __init__(self, validator: RequestValidator, guard: IdempotencyGuard):
        self.validator = validator
        self.guard = guard

    def create_episode(self, request: UniversalEpisodeWriteRequest) -> UniversalWriteResponse:
        dto = self.validator.validate_episode(request)
        return self.guard.execute(request.idempotency_key, lambda: episodic_service.create_episode(dto))

    def create_knowledge(self, request: UniversalKnowledgeWriteRequest) -> UniversalWriteResponse:
        dto = self.validator.validate_knowledge(request)
        return self.guard.execute(request.knowledge_id, lambda: semantic_service.create_knowledge(dto))


facade = MemoryFacadeDemo(validator, guard)


## Сценарий: создание эпизода → запись знания

Ниже смоделируем входящие данные, прогоним их через пайплайн и посмотрим на логи хранилищ и итоговые ответы.


In [ ]:
now = datetime.utcnow()

episode_request = UniversalEpisodeWriteRequest(
    idempotency_key="episode_support_001",
    context=EpisodeContext(user_id="user_123", agent_id="support_agent", session_id="session_007"),
    scope="user_private",
    episode_type="interaction",
    title="Поддержка по биллингу",
    trajectory=[
        EpisodeTrajectoryStep(
            step_id="1",
            timestamp=now,
            role="user",
            action="message",
            content="Мне нужна помощь с двойным списанием",
        ),
        EpisodeTrajectoryStep(
            step_id="2",
            timestamp=now,
            role="agent",
            action="response",
            content="Уточните, пожалуйста, номер аккаунта",
        ),
    ],
    outcome="Проблема решена",
    importance=0.85,
)

knowledge_request = UniversalKnowledgeWriteRequest(
    knowledge_id="kb_priority_support",
    knowledge="Премиум-пользователям доступна приоритетная обработка биллинговых инцидентов",
    source="system",
    confidence=0.95,
    agent_id="support_agent",
    supporting_evidence=[episode_request.idempotency_key],
    access_scope="team_shared",
)

print("--- Создание эпизода ---")
episode_response = facade.create_episode(episode_request)
print(pretty(episode_response.__dict__))

print("\n--- Повторный вызов (идемпотентность) ---")
episode_response_second = facade.create_episode(episode_request)
print(pretty(episode_response_second.__dict__))

print("\n--- Создание знания ---")
knowledge_response = facade.create_knowledge(knowledge_request)
print(pretty(knowledge_response.__dict__))


## Журнал артефактов

После выполнения можно проверить содержимое in-memory хранилищ, чтобы увидеть, какие структуры туда попали и какие поля сохраняются по контракту.


In [ ]:
print("Документы Mongo:")
for doc in mongo.documents:
    print(pretty(doc))

print("\nВекторы Qdrant:")
for vec in qdrant.vectors:
    print(pretty(vec))
